# Notebook 03: Ingredient Parsing

**Phase:** Semantic Classification (Thesis Plan Deliverable 1)

**Purpose:** Parse product-level ingredient lists into individual ingredient tokens, creating an ingredient-level dataset for semantic classification.

## Overview

The semantic classification task requires **ingredient-level** annotations, not product-level.
This notebook takes the cleaned ingredient lists and splits them into individual ingredients.

## Workflow

1. Load cleaned data from Notebook 02
2. Parse each ingredient list into individual ingredient tokens
3. Build an ingredient-level dataset (one row per ingredient, linked to its product)
4. Collect unique ingredient vocabulary for labeling in Notebook 04
5. Save parsed ingredient dataset

In [1]:
import os
import pandas as pd
from collections import Counter

from utils.data_utils import load_cleaned_data, get_data_directories, save_metadata
from utils.semantic_utils import parse_ingredient_list, get_semantic_category_list
from utils.text_processing import clean_html

print("Imports successful.")

Imports successful.


## 1. Load Cleaned Data

In [2]:
# Get project directories
dirs = get_data_directories()
print(f"Project base: {dirs['base']}")

# Load cleaned dataset from Notebook 02
df = load_cleaned_data(filepath=os.path.join(dirs['processed'], 'cleaned_dataset.csv'))
print(f"Loaded {len(df)} products")
print(f"Columns: {list(df.columns)}")

Project base: /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION
Loaded 1057 products
Columns: ['code', 'brands', 'product_name_en', 'ingredients_text_en', 'ingredients', 'ingredients_tags', 'allergens_tags', 'traces_tags', 'categories', 'countries_tags', 'ingredients_cleaned', 'ingredient_tokens_raw', 'ingredient_tokens']


## 2. Parse Ingredient Lists

Use `parse_ingredient_list()` from `semantic_utils` to split each product's ingredient string into individual ingredients.

In [3]:
# Parse ingredient lists into individual tokens
df['ingredient_parsed'] = df['ingredients_text_en'].apply(
    lambda x: parse_ingredient_list(clean_html(str(x)))
)

# Show parsing results
print(f"Products with parsed ingredients: {(df['ingredient_parsed'].str.len() > 0).sum()}")
print(f"Average ingredients per product: {df['ingredient_parsed'].str.len().mean():.1f}")

# Display examples
print("\nParsing examples (first 3):")
for i in range(min(3, len(df))):
    print(f"\nProduct {i}: {df.iloc[i]['product_name_en']}")
    print(f"  Raw: {str(df.iloc[i]['ingredients_text_en'])[:150]}...")
    print(f"  Parsed: {df.iloc[i]['ingredient_parsed'][:15]}")

Products with parsed ingredients: 1057
Average ingredients per product: 17.6

Parsing examples (first 3):

Product 0: Mogu litchi
  Raw: water, lychee juice from concentrate 25%, coconut 25%, fructose, sucrose, acidifiers (citric acid, calcium lactate), lychee flavor, preservative e211,...
  Parsed: ['water', 'lychee juice from concentrate 25%', 'coconut 25%', 'fructose', 'sucrose', 'acidifiers', 'citric acid', 'calcium lactate', 'lychee flavor', 'preservative e211', 'gelling agent: gellan gum', 'color: e129 - 0.0001%', 'this dye may have adverse effects on activity', 'attention in children']

Product 1: Oishi Prawn Crackers (Sweet Extra Flavor)
  Raw: wheat, tapioca starch, vegetable oil (may consist of coconut oil and/or palm olein), corn starch, shrimps, iodized salt, sugar, flavor enhancers (mono...
  Parsed: ['wheat', 'tapioca starch', 'vegetable oil', 'may consist of coconut oil and/or palm olein', 'corn starch', 'shrimps', 'iodized salt', 'sugar', 'flavor enhancers', 'monosodium

## 3. Build Ingredient-Level Dataset

Create an expanded dataset where each row is a single ingredient, linked back to its source product.

In [4]:
# Expand into ingredient-level rows
ingredient_rows = []
for idx, row in df.iterrows():
    ingredients = row['ingredient_parsed']
    if not ingredients:
        continue
    for pos, ingredient in enumerate(ingredients):
        ingredient_rows.append({
            'product_code': row['code'],
            'product_name': row.get('product_name_en', ''),
            'brand': row.get('brands', ''),
            'ingredient': ingredient,
            'position': pos,  # order in the ingredient list
        })

ingredient_df = pd.DataFrame(ingredient_rows)
print(f"Ingredient-level dataset: {len(ingredient_df)} individual ingredients")
print(f"  from {ingredient_df['product_code'].nunique()} products")
print(f"  with {ingredient_df['ingredient'].nunique()} unique ingredient terms")

ingredient_df.head(10)

Ingredient-level dataset: 18623 individual ingredients
  from 1057 products
  with 4664 unique ingredient terms


,product_code,product_name,brand,ingredient,position
0,8850389100684,Mogu litchi,Sappe Public Company Limited,water,0
1,8850389100684,Mogu litchi,Sappe Public Company Limited,lychee juice from concentrate 25%,1
2,8850389100684,Mogu litchi,Sappe Public Company Limited,coconut 25%,2
3,8850389100684,Mogu litchi,Sappe Public Company Limited,fructose,3
4,8850389100684,Mogu litchi,Sappe Public Company Limited,sucrose,4
5,8850389100684,Mogu litchi,Sappe Public Company Limited,acidifiers,5
6,8850389100684,Mogu litchi,Sappe Public Company Limited,citric acid,6
7,8850389100684,Mogu litchi,Sappe Public Company Limited,calcium lactate,7
8,8850389100684,Mogu litchi,Sappe Public Company Limited,lychee flavor,8
9,8850389100684,Mogu litchi,Sappe Public Company Limited,preservative e211,9


## 4. Ingredient Vocabulary Analysis

Identify the most common ingredients and the long tail of rare ones.

In [5]:
# Count ingredient frequencies
ingredient_counts = Counter(ingredient_df['ingredient'])

# Most common ingredients
print("Most common ingredients:")
for ingredient, count in ingredient_counts.most_common(30):
    print(f"  {ingredient:30s}  {count:4d} products")

print(f"\nVocabulary stats:")
print(f"  Unique ingredients: {len(ingredient_counts)}")
print(f"  Ingredients appearing only once: {sum(1 for c in ingredient_counts.values() if c == 1)}")
print(f"  Coverage of top-50 ingredients: {sum(c for _, c in ingredient_counts.most_common(50)) / len(ingredient_df):.1%}")

Most common ingredients:
  sugar                            635 products
  water                            333 products
  iodized salt                     313 products
  vegetable oil                    240 products
  citric acid                      210 products
  salt                             208 products
  spices                           193 products
  wheat flour                      173 products
  emulsifier                       171 products
  monosodium glutamate             160 products
  palm oil                         130 products
  stabilizer                       127 products
  artificial flavors               112 products
  antioxidant                      112 products
  potassium sorbate                107 products
  maltodextrin                     105 products
  natural                          103 products
  acidity regulator                 99 products
  cocoa powder                      99 products
  sodium bicarbonate                98 products
  lodized salt 

## 5. Save Ingredient Dataset

Save the parsed ingredient dataset for labeling in Notebook 04.

In [6]:
# Create output path
ingredient_path = os.path.join(dirs['interim'], 'parsed_ingredients.csv')
ingredient_df.to_csv(ingredient_path, index=False)
print(f"Saved parsed ingredients to: {ingredient_path}")

# Also export the unique ingredient vocabulary for manual annotation
vocab_df = pd.DataFrame([
    {'ingredient': ing, 'count': cnt}
    for ing, cnt in ingredient_counts.most_common()
])
vocab_path = os.path.join(dirs['interim'], 'ingredient_vocabulary.csv')
vocab_df.to_csv(vocab_path, index=False)
print(f"Saved ingredient vocabulary ({len(vocab_df)} unique terms) to: {vocab_path}")

Saved parsed ingredients to: /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/data/interim/parsed_ingredients.csv
Saved ingredient vocabulary (4664 unique terms) to: /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/data/interim/ingredient_vocabulary.csv


## 6. Summary Stats

In [7]:
print("=" * 50)
print("INGREDIENT PARSING SUMMARY")
print("=" * 50)
print(f"  Products parsed:            {len(df)}")
print(f"  Products with ingredients:  {(df['ingredient_parsed'].str.len() > 0).sum()}")
print(f"  Total ingredient instances: {len(ingredient_df)}")
print(f"  Unique ingredient terms:    {len(ingredient_counts)}")
print(f"  Avg ingredients/product:    {df['ingredient_parsed'].str.len().mean():.1f}")
print(f"  Ingredients once only:      {sum(1 for c in ingredient_counts.values() if c == 1)}")
print()
print("📋 Next step: Run 04_semantic_labeling.ipynb to annotate these")

INGREDIENT PARSING SUMMARY
  Products parsed:            1057
  Products with ingredients:  1057
  Total ingredient instances: 18623
  Unique ingredient terms:    4664
  Avg ingredients/product:    17.6
  Ingredients once only:      3282

📋 Next step: Run 04_semantic_labeling.ipynb to annotate these
